# 03  -   CNN: Clasificación de Mosquito


## 1. Carga de Librerías


In [49]:
import tensorflow as tf
import numpy as np

from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import mixed_precision
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Input, Rescaling, 
                                     Conv2D, MaxPooling2D, 
                                     Dense, Dropout, 
                                     BatchNormalization, GlobalAveragePooling2D)
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau)

## 1.2 Carga de Datos a Utilizar

In [50]:
train_ds = image_dataset_from_directory(
    "../data/processed/entrenamiento",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED
)

class_names = train_ds.class_names

print(class_names)

Found 5718 files belonging to 4 classes.
['aegypti', 'albopictus', 'anopheles', 'culex']


In [51]:
mixed_precision.set_global_policy("mixed_float16")

IMG_SIZE = (224, 224)


BATCH_SIZE = 16

EPOCHS = 30

SEED = 42

print(
    tf.config.list_physical_devices("GPU")
)

train_ds = image_dataset_from_directory(
    "../data/processed/entrenamiento",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED
)

val_ds = image_dataset_from_directory(
    "../data/processed/validacion",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED
)

test_ds = image_dataset_from_directory(
    "../data/processed/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Found 5718 files belonging to 4 classes.
Found 1141 files belonging to 4 classes.
Found 1153 files belonging to 4 classes.


### Control de Pesos

In [52]:
labels = []

for _, y in train_ds:

    labels.extend(
        y.numpy()
    )

labels = np.array(labels)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)

class_weights = dict(
    enumerate(class_weights)
)

print(class_weights)

{0: np.float64(0.6062340966921119), 1: np.float64(0.836943793911007), 2: np.float64(3.7033678756476682), 3: np.float64(1.1291469194312795)}


## 1.3 Guardado de Clases

In [53]:
class_names = sorted([
    p.name
    for p in Path("../data/processed/entrenamiento").iterdir()
    if p.is_dir()
])

print(class_names)

NUM_CLASSES = len(class_names)

print("Número de clases:", NUM_CLASSES)

['aegypti', 'albopictus', 'anopheles', 'culex']
Número de clases: 4


## 1.4 Optimización de Pipeline

In [54]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)

val_ds = val_ds.prefetch(AUTOTUNE)

test_ds = test_ds.prefetch(AUTOTUNE)

In [55]:
data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip(
        "horizontal"
    ),

    tf.keras.layers.RandomRotation(
        0.15
    ),

    tf.keras.layers.RandomZoom(
        0.15
    )

])

## 2. Creación del Modelo

En este caso vamos a dos modelos, uno siendo el CNN que se vio en clase y el otro siendo un modelo con MobileNetV2

### 2.1.1 Creación de Modelo CNN

In [56]:
model_cnn = Sequential([

    Input(shape=(224,224,3)),

    Rescaling(1./255),

    data_augmentation,

    # Bloque 1
    Conv2D(
        32,
        (3,3),
        activation="relu"
    ),

    BatchNormalization(),

    MaxPooling2D((2,2)),

    # Bloque 2
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),

    BatchNormalization(),

    MaxPooling2D((2,2)),

    # Bloque 3
    Conv2D(
        128,
        (3,3),
        activation="relu"
    ),

    BatchNormalization(),

    MaxPooling2D((2,2)),

Conv2D(
    256,
    (3,3),
    activation="relu"
),

BatchNormalization(),

MaxPooling2D((2,2)),

GlobalAveragePooling2D(),

Dense(
    256,
    activation="relu"
),

Dropout(
    0.5
),

Dense(
    NUM_CLASSES,
    activation="softmax",
    dtype="float32"
)

])

### 2.1.2 Compilación

In [57]:
model_cnn.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

model_cnn.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_3 (Rescaling)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_5 (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 222, 222, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 109, 109, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 52, 52, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_14          │ (None, 24, 24, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 457,156 (1.74 MB)

 Trainable params: 456,196 (1.74 MB)

 Non-trainable params: 960 (3.75 KB)

### 2.1.3 Callbacks

In [58]:
callbacks = [

    EarlyStopping(

        monitor="val_loss",

        patience=5,

        restore_best_weights=True

    ),

    ReduceLROnPlateau(

        monitor="val_loss",

        factor=0.5,

        patience=2

    )

]

### 2.1.2 Entrenamiento CNN

In [59]:
history_cnn = model_cnn.fit(

    train_ds,

    validation_data=val_ds,

    epochs=EPOCHS,

    callbacks=callbacks

)


Epoch 1/30


/home/sebas/Vector_IA/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


358/358 ━━━━━━━━━━━━━━━━━━━━ 38s 97ms/step - accuracy: 0.5701 - loss: 0.8929 - val_accuracy: 0.5259 - val_loss: 1.2476 - learning_rate: 0.0010
Epoch 2/30
358/358 ━━━━━━━━━━━━━━━━━━━━ 34s 96ms/step - accuracy: 0.5923 - loss: 0.7826 - val_accuracy: 0.6512 - val_loss: 0.9142 - learning_rate: 0.0010
Epoch 3/30
358/358 ━━━━━━━━━━━━━━━━━━━━ 35s 98ms/step - accuracy: 0.6611 - loss: 0.7056 - val_accuracy: 0.2217 - val_loss: 3.3300 - learning_rate: 0.0010
Epoch 4/30
358/358 ━━━━━━━━━━━━━━━━━━━━ 37s 104ms/step - accuracy: 0.7331 - loss: 0.6045 - val_accuracy: 0.7082 - val_loss: 0.5976 - learning_rate: 0.0010
Epoch 5/30
358/358 ━━━━━━━━━━━━━━━━━━━━ 36s 101ms/step - accuracy: 0.7658 - loss: 0.5510 - val_accuracy: 0.5898 - val_loss: 0.8779 - learning_rate: 0.0010
Epoch 6/30
358/358 ━━━━━━━━━━━━━━━━━━━━ 35s 97ms/step - accuracy: 0.7711 - loss: 0.5272 - val_accuracy: 0.5311 - val_loss: 1.3628 - learning_rate: 0.0010
Epoch 7/30
358/358 ━━━━━━━━━━━━━━━━━━━━ 35s 97ms/step - accuracy: 0.8017 - loss: 0.46

In [44]:
from pathlib import Path

for clase in class_names:

    n = len(
        list(
            Path("../data/processed/entrenamiento")
            .joinpath(clase)
            .glob("*")
        )
    )

    print(clase, n)

aegypti 2358
albopictus 1708
anopheles 386
culex 1266


In [45]:
for images, labels in train_ds.take(1):

    print(
        images.shape
    )

    print(
        labels[:20]
    )

(16, 224, 224, 3)
tf.Tensor([0 0 0 3 1 1 0 0 3 3 0 1 0 1 1 3], shape=(16,), dtype=int32)
